In [2]:
import numpy as np
import pyspedas
import fipcore as fpc
import os
import h5py

In [3]:
trange = ['2015-12-09/05:03:55', '2015-12-09/05:03:59']

# Plasma Beta Calculation

In [60]:
from pyspedas.projects import mms
from pyspedas import tplot_rename, get_data
from fipcore.utils.helper_utils import downsample_cad


def compute_beta_profiles(trange, species='e', probe='1', data_rate='brst', level='l2'):

    spec_tag = 'des' if species == 'e' else 'dis'

    # Variables to load:
    den_var = f'mms{probe}_{spec_tag}_numberdensity_{data_rate}'
    t_para_var = f'mms{probe}_{spec_tag}_temppara_{data_rate}'
    t_perp_var = f'mms{probe}_{spec_tag}_tempperp_{data_rate}'
    bvec_var = f'mms{probe}_fgm_b_gse_{data_rate}_{level}'

    # Loading in density and temperature data from fpi
    mms.fpi(trange= trange, probe=probe, data_rate=data_rate, level=level,
        datatype=f'{spec_tag}-moms', 
        time_clip=True, varnames=[den_var, t_para_var, t_perp_var], 
        get_support_data=get_support_data, no_update=no_update)

    # Loading in magnetic field data from fgm
    mms.fgm(trange=trange, probe=probe, data_rate=data_rate, level=level,
        varnames=bvec_var, time_clip=True, 
        get_support_data=get_support_data, no_update=no_update)

    # Renaming the tplot variables
    tplot_rename(den_var, 'den')
    tplot_rename(t_para_var, 't_para')
    tplot_rename(t_perp_var, 't_perp')
    tplot_rename(bvec_var, 'bvec')

    # Downsampling the fgm data to the fpi cadence
    downsample_cad('bvec', 'den', trange, newname='bvec_ds')

    # Extracting data from tplot variables
    _, den = get_data('den')
    _, t_para = get_data('t_para')
    _, t_perp = get_data('t_perp')
    _, bvec = get_data('bvec_ds') # bvec is [Bx, By, Bz, Bmag]
    bmag = bvec[:, 3]

    # Calling the helper functions to compute the individual beta profiles
    beta_par = compute_beta_parallel(den, t_para, bmag)
    beta_perp = compute_beta_perpendicular(den, t_perp, bmag)
    beta_tot = compute_beta_tot(den, t_para, t_perp, bmag)
    
    return beta_par, beta_perp, beta_tot

In [44]:
from scipy.constants import mu_0, e

def compute_beta_parallel(den, t_para, b_mag):
    """Plasma beta profile using parallel temperature."""
    p_para = (den * 1e6) * e * t_para
    p_mag = (b_mag * 1e-9)**2 / (2.0 * mu_0)
    return p_para / p_mag

def compute_beta_perpendicular(den, t_perp, b_mag):
    """Plasma beta profile using perpendicular temperature."""
    p_perp = (den * 1e6) * e * t_perp
    p_mag = (b_mag * 1e-9)**2 / (2.0 * mu_0)
    return p_perp / p_mag

def compute_beta_tot(den, t_para, t_perp, b_mag):
    """Plasma beta profile using average temperature."""
    t_avg = (t_para + 2.0 * t_perp) / 3.0
    p_total = (den * 1e6) * e * t_avg
    p_mag = (b_mag * 1e-9)**2 / (2.0 * mu_0)
    return p_total / p_mag